In [9]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [10]:


from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "val.jsonl"
    }
)

In [11]:
def format_example(example):
    return {
        "text": f"""
### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}
"""
    }

dataset = dataset.map(format_example)

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [13]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=False,

    gradient_checkpointing=True
)

In [15]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,   # ✅ THIS replaces tokenizer
    args=training_args,
)

In [16]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.369475
20,1.302542
30,0.626457
40,0.278182
50,0.133816
60,0.110217
70,0.077093
80,0.074260
90,0.065169
100,0.071322


TrainOutput(global_step=810, training_loss=0.10901130187658616, metrics={'train_runtime': 325.8342, 'train_samples_per_second': 9.944, 'train_steps_per_second': 2.486, 'total_flos': 1224063035080704.0, 'train_loss': 0.10901130187658616})

In [17]:
model.save_pretrained("adapters")
tokenizer.save_pretrained("adapters")

('adapters/tokenizer_config.json',
 'adapters/chat_template.jinja',
 'adapters/tokenizer.json')